# Analytic Log Expected Hypervolume Improvement

- Contributors: SebastianAment
- Last updated: September 8, 2026
- BoTorch version: 0.18.1+ (main, commit `d4b9fc655`)

> **Work in progress.** This notebook prototypes an analytic, log-space variant of
> Expected Hypervolume Improvement. The cells are committed **without outputs and have
> not been executed against the current BoTorch release** — expect to fix things. It is
> shared so that others can build on it. The closing section describes what is
> incomplete.

## Motivation

The LogEI family of acquisition functions addresses a numerical problem rather than a
modeling one. Expected Improvement underflows to exactly zero over most of the search
space in floating point, so its gradient vanishes and the optimizer has nothing to
follow; reformulating the same quantity in log space recovers a usable gradient
essentially everywhere. See
[Ament et al., *Unexpected Improvements to Expected Improvement for Bayesian
Optimization*, NeurIPS 2023](https://arxiv.org/abs/2310.20708).

The multi-objective analytic case has the same problem, and arguably a worse version
of it. Analytic EHVI decomposes the expected hypervolume improvement over the cells of
a box decomposition of the non-dominated region, and within each cell it forms a
*product* of per-outcome factors $\Psi$ and $\nu$ (equations 19 and 25 of
[Yang et al. 2019]). Products of small numbers underflow much faster than individual
small numbers, so the regime where the acquisition value is exactly zero is
correspondingly larger.

BoTorch open-sources the analytic `ExpectedHypervolumeImprovement`, and the Monte
Carlo based `qLogExpectedHypervolumeImprovement` and
`qLogNoisyExpectedHypervolumeImprovement`. What is missing is the analytic function in
log space, which is what this notebook works out.

The bulk of the work is one function: a numerically stable evaluation of
$\log \Psi_{\text{diff}}$, the log of the difference of two $\Psi$ terms. Everything
else follows from it.

In [ ]:
import matplotlib.pyplot as plt
import torch

from botorch.acquisition.analytic import _log_ei_helper
from botorch.acquisition.multi_objective.analytic import (
    ExpectedHypervolumeImprovement,
)
from botorch.utils.probability.utils import log_ndtr, ndtr, phi
from botorch.utils.safe_math import logdiffexp
from botorch.utils.transforms import t_batch_mode_transform
from torch import Tensor

torch.set_default_dtype(torch.double)

## The naive baseline and where it breaks

The quantity of interest is the difference

$$\Psi_{\text{diff}} = \Psi(l, l, \mu, \sigma) - \Psi(l, u, \mu, \sigma),$$

where $l$ and $u$ are the lower and upper bounds of a cell along one outcome. The
definition of $\Psi$ is reproduced below; it matches the `psi` method on the
open-source `ExpectedHypervolumeImprovement`.

In [ ]:
def psi(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    """The Psi function of Yang et al. (2019), eq. 19.

    Psi(l, u, mu, sigma) = sigma * phi((u - mu) / sigma)
                           + (mu - l) * (1 - Phi((u - mu) / sigma))
    """
    u_upper = (upper - mu) / sigma
    return sigma * phi(u_upper) + (mu - lower) * (1 - ndtr(u_upper))


def psi_diff_naive(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    return psi(lower, lower, mu, sigma) - psi(lower, upper, mu, sigma)

In [ ]:
lower = torch.tensor(-1.2)
upper = torch.tensor(0.8)
sigma = torch.tensor(1e3)
mu = torch.linspace(-100.0, 100.0, 201)

naive = psi_diff_naive(lower, upper, mu, sigma)
log_naive = naive.log()

underflowed = log_naive.isinf() | log_naive.isnan()
print(f"naive log(psi_diff) is non-finite for {int(underflowed.sum())} of {len(mu)} points")
if underflowed.any():
    print(f"first failure at mu = {mu[underflowed][0].item():.2f}")

## Deriving a stable expression

Write the standardized bounds

$$u_l = \frac{l - \mu}{\sigma}, \qquad u_u = \frac{u - \mu}{\sigma},$$

and let $h(t) = \varphi(t) + t\,\Phi(t)$ be the EI helper, whose logarithm BoTorch
already computes stably as `_log_ei_helper`.

Starting from the definition of $\Psi$ and substituting,

$$\frac{\Psi(l, u, \mu, \sigma)}{\sigma}
   = \varphi(-u_u) - u_l\,\Phi(-u_u).$$

Adding and subtracting $u_u \Phi(-u_u)$ regroups this into the EI helper plus a
remainder:

$$\frac{\Psi(l, u, \mu, \sigma)}{\sigma}
   = \underbrace{\bigl[\varphi(-u_u) - u_u\,\Phi(-u_u)\bigr]}_{h(-u_u)}
     + (u_u - u_l)\,\Phi(-u_u)
   = h(-u_u) + \frac{u - l}{\sigma}\,\Phi(-u_u).$$

The other term is simpler, because $u = l$ collapses the two arguments:

$$\frac{\Psi(l, l, \mu, \sigma)}{\sigma} = h(-u_l).$$

So the difference is

$$\frac{\Psi_{\text{diff}}}{\sigma}
   = h(-u_l) - \left[ h(-u_u) + \frac{u-l}{\sigma}\,\Phi(-u_u) \right],$$

which is a difference of two positive quantities. Taking logarithms and using
`logsumexp` for the bracketed sum and `logdiffexp` for the outer subtraction:

$$\log \Psi_{\text{diff}} = \log \sigma
  + \operatorname{logdiffexp}\Bigl(
      \operatorname{logsumexp}\bigl[\log h(-u_u),\;
        \log\tfrac{u-l}{\sigma} + \log\Phi(-u_u)\bigr],\;
      \log h(-u_l)
    \Bigr).$$

Every term on the right is computed by an existing stable primitive: `_log_ei_helper`
for $\log h$, `log_ndtr` for $\log \Phi$, and `logsumexp` / `logdiffexp` from
`botorch.utils.safe_math`. Note that $\log\Phi(-u_u)$ is evaluated directly through
`log_ndtr` rather than as $\log(1 - \Phi(u_u))$, which would lose all precision in
exactly the tail regime we care about.

In [ ]:
def log_psi_diff(
    lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor
) -> Tensor:
    """Numerically stable evaluation of log(psi(l, l, mu, s) - psi(l, u, mu, s)).

    Args:
        lower: `num_cells x m`-dim tensor of lower cell bounds.
        upper: `num_cells x m`-dim tensor of upper cell bounds.
        mu: `batch_shape x 1 x m`-dim tensor of means.
        sigma: `batch_shape x 1 x m`-dim tensor of standard deviations.

    Returns:
        A `batch_shape x num_cells x m`-dim tensor of log values.
    """
    u_lower = (lower - mu) / sigma
    u_upper = (upper - mu) / sigma
    log_u_diff = ((upper - lower) / sigma).log()

    # log of h(-u_upper) + ((u - l) / sigma) * Phi(-u_upper)
    log_subtrahend = torch.logsumexp(
        torch.stack(
            [
                _log_ei_helper(-u_upper),
                log_u_diff + log_ndtr(-u_upper),
            ],
            dim=0,
        ),
        dim=0,
    )
    # log of h(-u_lower), the larger of the two terms
    log_minuend = _log_ei_helper(-u_lower)

    return sigma.log() + logdiffexp(log_a=log_subtrahend, log_b=log_minuend)

### Agreement and range

Two checks. First, that the log form agrees with the naive one wherever the latter is
representable. Second, that it stays finite where the naive one does not.

In [ ]:
log_stable = log_psi_diff(lower, upper, mu, sigma)

finite = ~underflowed
relative_error = (
    (log_stable[finite] - log_naive[finite]).abs()
    / log_naive[finite].abs().clamp_min(1e-12)
)
print(f"max relative error where naive is representable: {relative_error.max():.3e}")
print(f"log_psi_diff non-finite anywhere: {bool(log_stable.isfinite().logical_not().any())}")

In [ ]:
u_lower_axis = (lower - mu) / sigma

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(u_lower_axis, log_stable, lw=2, label="log_psi_diff (stable)")
ax.plot(u_lower_axis, log_naive, lw=2, ls=":", label="log(psi_diff) (naive)")
if underflowed.any():
    first = int(underflowed.nonzero()[0])
    ax.axvline(
        u_lower_axis[first], color="red", ls="--", lw=1,
        label="naive underflow",
    )
ax.set_xlabel("u_lower = (lower - mu) / sigma")
ax.set_ylabel("log value")
ax.set_title("The naive form underflows; the log form does not")
ax.legend()

### Gradients

This is the stricter test, and the one that turned up real bugs during development. An
expression can return a finite value and still produce `NaN` gradients, typically
because a masked-out branch of a `where` is evaluated anyway and its derivative is
undefined. Single precision with a large `sigma` is where this shows up first.

In [ ]:
def check_gradients(fn, dtype: torch.dtype, sigma_value: float, label: str) -> None:
    n = 500
    mu_g = torch.linspace(0.0, 100.0, n, dtype=dtype, requires_grad=True)
    sigma_g = torch.full((n,), sigma_value, dtype=dtype, requires_grad=True)
    lower_g = torch.tensor(-1.2, dtype=dtype)
    upper_g = torch.tensor(0.8, dtype=dtype)

    value = fn(lower_g, upper_g, mu_g, sigma_g)
    value.sum().backward()

    def bad(t):
        return bool(t.isnan().any() or t.isinf().any())

    print(
        f"{label:>28} | value {'BAD' if bad(value) else ' ok'} "
        f"| d/dmu {'BAD' if bad(mu_g.grad) else ' ok'} "
        f"| d/dsigma {'BAD' if bad(sigma_g.grad) else ' ok'}"
    )


for dtype, sigma_value in [(torch.float32, 1e4), (torch.float64, 1e4)]:
    name = "float32" if dtype is torch.float32 else "float64"
    check_gradients(
        lambda l, u, m, s: psi_diff_naive(l, u, m, s).log(),
        dtype, sigma_value, f"naive, {name}",
    )
    check_gradients(log_psi_diff, dtype, sigma_value, f"log_psi_diff, {name}")

## The $\nu$ term

The second factor is straightforward, since it is already a product rather than a
difference — taking logs turns it into a sum with no cancellation.

$$\log \nu(l, u, \mu, \sigma) = \log(u - l) + \log \Phi\!\left(\frac{\mu - u}{\sigma}\right).$$

In [ ]:
def log_nu(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    """Logarithm of the Nu function of Yang et al. (2019), eq. 25."""
    return log_ndtr((mu - upper) / sigma) + (upper - lower).log()


check_gradients(log_nu, torch.float32, 1.2, "log_nu, float32")
check_gradients(log_nu, torch.float64, 1e4, "log_nu, float64")

## Assembling the acquisition function

With both factors available in log space, the rest of the analytic EHVI computation
carries over. For each cell the contribution is a product over outcomes of either
$\Psi_{\text{diff}}$ or $\nu$, taken over all $2^m$ combinations; in log space the
product becomes a sum, and the outer sums over cells and combinations become
`logsumexp`.

Subclassing the open-source `ExpectedHypervolumeImprovement` means the partitioning and
cell-bound bookkeeping is inherited, and only `forward` needs to change.

In [ ]:
class LogExpectedHypervolumeImprovement(ExpectedHypervolumeImprovement):
    r"""Analytic log Expected Hypervolume Improvement.

    Computes the same quantity as `ExpectedHypervolumeImprovement`, but in log space,
    so that the value and its gradient remain informative in regimes where the
    non-log version underflows to exactly zero.

    Supports `q = 1` and no pending points, matching the analytic parent class.
    """

    @t_batch_mode_transform()
    def forward(self, X: Tensor) -> Tensor:
        posterior = self.model.posterior(
            X, posterior_transform=self.posterior_transform
        )
        mu = posterior.mean
        sigma = posterior.variance.clamp_min(1e-9).sqrt()

        # The upper bounds contain infs, which are not differentiable.
        cell_upper_bounds = self.cell_upper_bounds.clamp_max(
            1e10 if X.dtype == torch.double else 1e8
        )

        log_psi_diff_values = log_psi_diff(
            lower=self.cell_lower_bounds,
            upper=cell_upper_bounds,
            mu=mu,
            sigma=sigma,
        )
        log_nu_values = log_nu(
            lower=self.cell_lower_bounds,
            upper=cell_upper_bounds,
            mu=mu,
            sigma=sigma,
        )

        # batch_shape x num_cells x 2 x m
        stacked_factors = torch.stack([log_psi_diff_values, log_nu_values], dim=-2)

        # Take the cross product across outcomes: batch_shape x num_cells x 2^m x m
        all_log_factors = stacked_factors.gather(
            dim=-2,
            index=self._cross_product_indices.expand(
                stacked_factors.shape[:-2] + self._cross_product_indices.shape
            ),
        )

        # Product over outcomes becomes a sum in log space; the sums over cells and
        # over the cross product become logsumexp.
        return torch.logsumexp(all_log_factors.sum(dim=-1), dim=(-2, -1))

### Comparison against the non-log version

Where analytic EHVI is accurate the two should agree after exponentiation. Where EHVI
has underflowed to exactly zero, LogEHVI should still return a finite value with a
non-zero gradient — which is the entire point.

In [ ]:
from botorch.models import SingleTaskGP
from botorch.models.model_list_gp_regression import ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.test_functions.multi_objective import BraninCurrin
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.sampling import draw_sobol_samples
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood

from botorch.fit import fit_gpytorch_mll

torch.manual_seed(0)

problem = BraninCurrin(negate=True)
d, m = problem.dim, problem.num_objectives
noise_se = torch.full((m,), 1e-2)

train_X = draw_sobol_samples(bounds=problem.bounds, n=16, q=1).squeeze(1)
train_Y = problem(train_X) + torch.randn(16, m) * noise_se

models = [
    SingleTaskGP(
        train_X=train_X,
        train_Y=train_Y[..., i : i + 1],
        train_Yvar=torch.full_like(train_Y[..., i : i + 1], noise_se[i] ** 2),
        input_transform=Normalize(d=d, bounds=problem.bounds),
        outcome_transform=Standardize(m=1),
    )
    for i in range(m)
]
model = ModelListGP(*models)
fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))

with torch.no_grad():
    pred = model.posterior(train_X).mean
partitioning = FastNondominatedPartitioning(ref_point=problem.ref_point, Y=pred)

ehvi = ExpectedHypervolumeImprovement(
    model=model, ref_point=problem.ref_point, partitioning=partitioning
)
log_ehvi = LogExpectedHypervolumeImprovement(
    model=model, ref_point=problem.ref_point, partitioning=partitioning
)

In [ ]:
test_X = draw_sobol_samples(bounds=problem.bounds, n=512, q=1).squeeze(1)

with torch.no_grad():
    ehvi_values = ehvi(test_X.unsqueeze(-2))
    log_ehvi_values = log_ehvi(test_X.unsqueeze(-2))

positive = ehvi_values > 0
agreement = (log_ehvi_values[positive].exp() - ehvi_values[positive]).abs()
relative = agreement / ehvi_values[positive]

print(f"points where EHVI is exactly zero:  {int((~positive).sum())} / {len(test_X)}")
print(f"max relative disagreement elsewhere: {relative.max():.3e}")
print(
    "LogEHVI at those points ranges over "
    f"[{log_ehvi_values[~positive].min():.1f}, {log_ehvi_values[~positive].max():.1f}]"
    if (~positive).any()
    else "EHVI was positive everywhere; try a farther-away test set."
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
order = ehvi_values.argsort()
ax.plot(
    log_ehvi_values[order], lw=1.5, label="LogEHVI",
)
with torch.no_grad():
    naive_log = ehvi_values[order].log()
ax.plot(naive_log, lw=1.5, ls=":", label="log(EHVI)")
ax.set_xlabel("test point (sorted by EHVI)")
ax.set_ylabel("log acquisition value")
ax.set_title("log(EHVI) truncates at -inf; LogEHVI keeps going")
ax.legend()

In [ ]:
# Gradients where EHVI has underflowed: LogEHVI should still provide a direction.
if (~positive).any():
    X_zero = test_X[~positive][:8].clone().requires_grad_(True)

    ehvi_grad_value = ehvi(X_zero.unsqueeze(-2)).sum()
    ehvi_grad_value.backward()
    ehvi_grad_norm = X_zero.grad.norm(dim=-1)

    X_zero_log = test_X[~positive][:8].clone().requires_grad_(True)
    log_ehvi(X_zero_log.unsqueeze(-2)).sum().backward()
    log_grad_norm = X_zero_log.grad.norm(dim=-1)

    print(f"{'|grad EHVI|':>14} | {'|grad LogEHVI|':>16}")
    for a, b in zip(ehvi_grad_norm, log_grad_norm):
        print(f"{a.item():>14.3e} | {b.item():>16.3e}")

## Status & open problems

The pieces above work, but this is not finished, and the gaps are all in the same
place: the branch structure of `log_psi_diff`.

- **The near-degenerate cell branch is incomplete.** When
  $(u - l)/\sigma$ is very small, the two $\Psi$ terms nearly cancel and `logdiffexp`
  loses precision. A Taylor expansion in $(u-l)/\sigma$ is the right fix — the
  second-order term works out to roughly
  $\log \Psi_{\text{diff}} \approx \log h(-u_l) + \log\bigl(1 - \exp(\kappa)\bigr)$
  with $\kappa = 2\log\frac{u-l}{\sigma} + \log\varphi(-u_l) - \log h(-u_l) - \log 2$ —
  but its accuracy was never characterized and the crossover point was never chosen.
  The likely shape of the solution is a two-branch structure analogous to the one
  inside `_log_ei_helper`.
- **The "above cell" branch is prototyped but not safe.** When $u_u > 0$ the
  $\varphi$ term changes sign and the expression above no longer applies directly; one
  can instead write the result as a difference of a $\Phi$ term and a $\varphi$ term
  using `log_prob_normal_in`. The version that exists is not masked, so it produces
  `NaN` gradients on the points it does not apply to. Getting the masking right is the
  same pattern as in the Matérn-at-zero fix: `masked_fill` the inactive branch before
  the `where`, not after.
- **$\Psi_{\text{diff}}$ is not symmetric about the cell center**, which is what makes
  a single unified branch awkward and is worth keeping in mind when designing the
  crossover logic.
- **No `q`-batch generalization.** This inherits the `q = 1` restriction of the
  analytic parent class. The Monte Carlo `qLogEHVI` already handles `q > 1`; whether
  the analytic factors can be composed with its machinery, or whether the analytic
  route is simply the wrong tool for batches, is open.
- **No benchmark.** There is no closed-loop comparison against
  `qLogExpectedHypervolumeImprovement` or `qLogNParEGO`. Given that the analytic
  version is restricted to `q = 1`, the honest question is whether it is worth having
  at all once the Monte Carlo versions exist — the case for it would be lower variance
  and lower cost in the sequential setting, and that has not been demonstrated.

---

## References

- S. Ament, S. Daulton, D. Eriksson, M. Balandat, E. Bakshy.
  [Unexpected Improvements to Expected Improvement for Bayesian Optimization](https://arxiv.org/abs/2310.20708).
  Advances in Neural Information Processing Systems 36, 2023.
- K. Yang, M. Emmerich, A. Deutz, T. Bäck. Efficient Computation of Expected
  Hypervolume Improvement Using Box Decomposition Algorithms. Journal of Global
  Optimization, 2019.
- S. Daulton, M. Balandat, E. Bakshy. Differentiable Expected Hypervolume Improvement
  for Parallel Multi-Objective Bayesian Optimization. Advances in Neural Information
  Processing Systems 33, 2020.